# PC Activity Analysis

Two analyses:
1. **Cluster PCs** into 'decreasing' vs 'non-decreasing' groups based on FR drop before reward (AllR condition)
2. **Speed-PSTH cross-correlation** per condition per group — two methods:
   - Method A: mean of per-cell lags
   - Method B: lag of group-averaged (max-normalised) PSTHs

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter1d
from scipy.signal import correlate
from sklearn.cluster import KMeans

from data_loader import SessionData

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1 · Load session

In [ ]:
# ── edit this path ──────────────────────────────────────────────────
RES_PATH = r'C:\path\to\new_res.mat'
# ───────────────────────────────────────────────────────────────────

sd = SessionData(RES_PATH)

print(f'Total units  : {sd.n_units}')
print(f'Neural units : {sd.n_neural_units}')
print(f'FPS          : {sd.fps:.2f} Hz')
print(f'Event keys   : {sd.event_keys}')
print(f'Trials/event : {sd.n_trials}')

## 2 · Identify units of interest

In [ ]:
# ── Purkinje cells ──────────────────────────────────────────────────
pc_idx = [i for i in range(sd.n_neural_units)
          if sd.cell_type_label[i] == 'PC']
print(f'PCs found: {len(pc_idx)}')

# ── Speed channel ───────────────────────────────────────────────────
speed_idx = next(
    (i for i in sd.beh_unit_indices
     if sd.cell_type_label[i] == 'Speed'),
    None
)
if speed_idx is None:
    raise RuntimeError('Speed channel not found in beh_unit_indices')
print(f'Speed index : {speed_idx}')

# ── AllR event key ──────────────────────────────────────────────────
allr_key = next(
    (k for k in sd.event_keys if 'allr' in k.lower() or k == 'AllR'),
    None
)
if allr_key is None:
    print('No AllR key found – available keys:', sd.event_keys)
    allr_key = sd.event_keys[0]   # fallback: first key
print(f'AllR key : {allr_key}')

## 3 · Analysis parameters

In [ ]:
WINDOW_SEC   = 3.0       # ± seconds around event
SMOOTH_SIG   = 3         # gaussian sigma in FR bins (≈ 50 ms at 60 Hz)

# Decrease-score windows (seconds before reward)
BASELINE_WIN = (-1.5, -0.5)   # baseline epoch
PRE_WIN      = (-0.5,  0.0)   # pre-reward epoch

# Cross-correlation max lag
MAX_LAG_SEC  = 2.0

---
# Part 1 · Cluster PCs into decreasing / non-decreasing

### 1a · Compute decrease score for each PC

**Decrease score** = `mean FR in pre-reward window / mean FR in baseline window`

Values < 1 indicate a drop; the more negative the ratio departs from 1 the stronger the decrease.

In [ ]:
def get_psth(unit_idx, event_key, smooth=SMOOTH_SIG, window=WINDOW_SEC):
    """Thin wrapper returning (mean, sem, time_ax)."""
    return sd.get_mean_psth(unit_idx, event_key, smooth, window)


def decrease_score(mean: np.ndarray, time_ax: np.ndarray,
                   baseline=BASELINE_WIN, pre=PRE_WIN) -> float:
    """Ratio pre / baseline. <1 → decreasing cell."""
    b_mask = (time_ax >= baseline[0]) & (time_ax <= baseline[1])
    p_mask = (time_ax >= pre[0])      & (time_ax <= pre[1])
    b_mean = float(mean[b_mask].mean()) if b_mask.any() else 1.0
    p_mean = float(mean[p_mask].mean()) if p_mask.any() else 1.0
    if b_mean == 0:
        return 1.0
    return p_mean / b_mean


# Collect PSTHs and scores
pc_psths   = {}   # unit_idx → (mean, sem, time_ax)
pc_scores  = {}   # unit_idx → float

for uid in pc_idx:
    mean, sem, tax = get_psth(uid, allr_key)
    pc_psths[uid]  = (mean, sem, tax)
    pc_scores[uid] = decrease_score(mean, tax)

scores_arr = np.array([pc_scores[u] for u in pc_idx])
print(f'Decrease scores — min:{scores_arr.min():.3f}  '
      f'max:{scores_arr.max():.3f}  '
      f'mean:{scores_arr.mean():.3f}')

### 1b · K-means cluster (k = 2)

In [ ]:
km = KMeans(n_clusters=2, n_init=50, random_state=42)
labels = km.fit_predict(scores_arr.reshape(-1, 1))

# Ensure label 0 = decreasing (lower score = more decreasing)
c0_mean = scores_arr[labels == 0].mean()
c1_mean = scores_arr[labels == 1].mean()
if c0_mean > c1_mean:            # flip so 0 = decreasing
    labels = 1 - labels

dec_idx    = [pc_idx[i] for i, l in enumerate(labels) if l == 0]
nondec_idx = [pc_idx[i] for i, l in enumerate(labels) if l == 1]

print(f'Decreasing     : {len(dec_idx)} PCs  '
      f'(score mean {scores_arr[labels==0].mean():.3f})')
print(f'Non-decreasing : {len(nondec_idx)} PCs  '
      f'(score mean {scores_arr[labels==1].mean():.3f})')

### 1c · Visualise clusters

In [ ]:
def norm_max(arr: np.ndarray) -> np.ndarray:
    """Divide by max absolute value (preserves sign/DC offset)."""
    mx = np.abs(arr).max()
    return arr / mx if mx > 0 else arr


def norm_peak(arr: np.ndarray) -> np.ndarray:
    """Shift min to 0, scale max to 1 (maps to [0, 1])."""
    arr = arr - arr.min()
    mx  = arr.max()
    return arr / mx if mx > 0 else arr


def make_heatmap_matrix(uid_list, norm_fn=norm_peak,
                        sort_by_score=True):
    """Return (matrix, time_ax, order) for heatmap; rows sorted by score."""
    rows, tax = [], None
    for uid in uid_list:
        mean, _, t = pc_psths[uid]
        tax = t
        rows.append(norm_fn(mean))
    mat = np.array(rows)   # (n_cells, n_time)
    if sort_by_score:
        order = np.argsort([pc_scores[u] for u in uid_list])
        mat   = mat[order]
    else:
        order = np.arange(len(uid_list))
    return mat, tax, order


# ── Figure layout ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig,
                        hspace=0.45, wspace=0.35)

tax_ref = pc_psths[pc_idx[0]][2]

# ---- Score histogram ------------------------------------------------
ax_hist = fig.add_subplot(gs[0, 0])
ax_hist.hist(scores_arr[labels==0], bins=15, alpha=0.7,
             color='#E53935', label='Decreasing')
ax_hist.hist(scores_arr[labels==1], bins=15, alpha=0.7,
             color='#1E88E5', label='Non-decreasing')
ax_hist.axvline(1.0, color='k', lw=1, ls='--', label='score=1')
ax_hist.set_xlabel('Decrease score (pre/baseline)')
ax_hist.set_ylabel('# PCs')
ax_hist.set_title('Score distribution')
ax_hist.legend(fontsize=8)

# ---- Mean PSTHs (z-scored per cell, then averaged) ------------------
ax_mean = fig.add_subplot(gs[1, 0])

def group_mean_sem(uid_list, norm_fn=norm_peak):
    rows = [norm_fn(pc_psths[u][0]) for u in uid_list]
    arr  = np.array(rows)
    return arr.mean(0), arr.std(0) / np.sqrt(len(rows))

for uid_list, color, label in [
    (dec_idx,    '#E53935', 'Decreasing'),
    (nondec_idx, '#1E88E5', 'Non-decreasing'),
]:
    if not uid_list:
        continue
    m, s = group_mean_sem(uid_list)
    ax_mean.plot(tax_ref, m, color=color, lw=1.8, label=label)
    ax_mean.fill_between(tax_ref, m-s, m+s, color=color, alpha=0.2)

ax_mean.axvline(0, color='gray', lw=1, ls='--')
ax_mean.set_xlabel('Time from reward (s)')
ax_mean.set_ylabel('Normalised FR')
ax_mean.set_title('Mean PSTH (peak-norm)')
ax_mean.legend(fontsize=8)

# ---- Heatmaps -------------------------------------------------------
for col, (uid_list, group_label, cmap) in enumerate([
    (dec_idx,    'Decreasing',     'RdBu_r'),
    (nondec_idx, 'Non-decreasing', 'RdBu_r'),
], start=1):
    if not uid_list:
        continue
    mat, tax, _ = make_heatmap_matrix(uid_list)
    ax = fig.add_subplot(gs[:, col])
    vmax = np.nanpercentile(np.abs(mat), 98)
    im = ax.imshow(
        mat, aspect='auto', cmap=cmap, vmin=-vmax, vmax=vmax,
        extent=[tax[0], tax[-1], len(uid_list), 0],
        interpolation='nearest',
    )
    ax.axvline(0, color='w', lw=0.8, ls='--')
    ax.set_xlabel('Time from reward (s)')
    ax.set_ylabel('PC (sorted by score)')
    ax.set_title(f'{group_label}\n(n={len(uid_list)})')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='Norm FR')

# ---- Per-cell score vs PSTH shape -----------------------------------
ax_sc = fig.add_subplot(gs[:, 3])
all_means_norm = np.array([norm_peak(pc_psths[u][0]) for u in pc_idx])
sc_order = np.argsort(scores_arr)
im2 = ax_sc.imshow(
    all_means_norm[sc_order], aspect='auto', cmap='RdBu_r',
    extent=[tax_ref[0], tax_ref[-1], len(pc_idx), 0],
    vmin=-1, vmax=1, interpolation='nearest',
)
ax_sc.axvline(0, color='w', lw=0.8, ls='--')
# Mark the boundary between clusters on the sorted heatmap
boundary = int((labels[sc_order] == 0).sum())
ax_sc.axhline(boundary, color='yellow', lw=1.5)
ax_sc.set_xlabel('Time from reward (s)')
ax_sc.set_ylabel('All PCs sorted by decrease score')
ax_sc.set_title('All PCs (yellow = cluster boundary)')
plt.colorbar(im2, ax=ax_sc, fraction=0.03, pad=0.02)

fig.suptitle(f'PC clustering — AllR condition  [{allr_key}]', fontsize=13)
plt.savefig('pc_clusters.png', bbox_inches='tight')
plt.show()
print('Saved pc_clusters.png')

---
# Part 2 · Speed–PSTH cross-correlation per condition

### 2a · Helper: compute lag between a PSTH and speed

In [ ]:
def xcorr_lag_ms(a: np.ndarray, b: np.ndarray,
                 time_ax: np.ndarray,
                 max_lag_sec: float = MAX_LAG_SEC
                 ) -> tuple[float, np.ndarray, np.ndarray]:
    """
    Cross-correlate `a` with `b` (both zero-meaned) and return the lag
    at the peak within ±max_lag_sec.

    Returns
    -------
    lag_ms   : float — positive → `a` is delayed relative to `b`
    lags_ms  : (n_lags,) — full lag axis in ms (for plotting)
    xc_norm  : (n_lags,) — normalised cross-correlation curve
    """
    dt    = float(time_ax[1] - time_ax[0])          # s / bin
    n     = len(a)
    a_zm  = a - a.mean()
    b_zm  = b - b.mean()

    xc    = correlate(a_zm, b_zm, mode='full')
    lags  = np.arange(-(n-1), n)                    # samples
    lags_s = lags * dt

    # restrict to ±max_lag_sec
    mask   = np.abs(lags_s) <= max_lag_sec
    xc_w   = xc[mask]
    lags_w = lags_s[mask]

    # normalise to [-1, 1]
    norm   = np.sqrt(np.dot(a_zm, a_zm) * np.dot(b_zm, b_zm))
    xc_w   = xc_w / norm if norm > 0 else xc_w

    peak_idx = int(np.argmax(np.abs(xc_w)))
    lag_ms   = float(lags_w[peak_idx]) * 1000.0     # s → ms

    return lag_ms, lags_w * 1000.0, xc_w

### 2b · Compute per-cell lags for every event condition

In [ ]:
# All lags:  {event_key: {unit_idx: lag_ms}}
all_lags: dict[str, dict[int, float]] = {}
# Cross-corr curves (for group averages):  {event_key: {unit_idx: (lags_ms, xc)}}
all_xc:   dict[str, dict[int, tuple]]  = {}

for key in sd.event_keys:
    all_lags[key] = {}
    all_xc[key]   = {}

    # Speed PSTH for this condition
    sp_mean, _, sp_tax = sd.get_mean_psth(speed_idx, key, SMOOTH_SIG, WINDOW_SEC)

    for uid in pc_idx:
        pc_mean, _, tax = sd.get_mean_psth(uid, key, SMOOTH_SIG, WINDOW_SEC)
        lag_ms, lags_ms, xc = xcorr_lag_ms(pc_mean, sp_mean, tax)
        all_lags[key][uid] = lag_ms
        all_xc[key][uid]   = (lags_ms, xc)

print('Per-cell lags computed for', len(sd.event_keys), 'conditions.')

### 2c · Method A — mean of per-cell lags

In [ ]:
groups = {
    'Decreasing':     dec_idx,
    'Non-decreasing': nondec_idx,
}
group_colors = {'Decreasing': '#E53935', 'Non-decreasing': '#1E88E5'}

# Summarise: mean ± SEM per group per condition
method_a: dict[str, dict[str, tuple]] = {}   # [group][key] = (mean_lag, sem_lag)

for gname, uid_list in groups.items():
    method_a[gname] = {}
    for key in sd.event_keys:
        lags = np.array([all_lags[key][u] for u in uid_list])
        method_a[gname][key] = (float(lags.mean()),
                                 float(lags.std() / np.sqrt(len(lags))))

# ── Plot bar chart ───────────────────────────────────────────────────
n_cond   = len(sd.event_keys)
x        = np.arange(n_cond)
bar_w    = 0.35

fig, ax = plt.subplots(figsize=(max(6, n_cond * 1.8), 5))

for gi, (gname, uid_list) in enumerate(groups.items()):
    means = [method_a[gname][k][0] for k in sd.event_keys]
    sems  = [method_a[gname][k][1] for k in sd.event_keys]
    offset = (gi - 0.5) * bar_w
    ax.bar(x + offset, means, bar_w, yerr=sems,
           color=group_colors[gname], alpha=0.85,
           capsize=4, label=gname, error_kw={'lw': 1.5})

ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(sd.event_keys, rotation=30, ha='right')
ax.set_ylabel('Lag (ms)  [+ = PC delayed vs speed]')
ax.set_title('Method A: mean of per-cell lags')
ax.legend()
plt.tight_layout()
plt.savefig('xcorr_method_a.png', bbox_inches='tight')
plt.show()

### 2d · Method B — lag of group-averaged PSTHs (max-normalised before averaging)

In [ ]:
method_b: dict[str, dict[str, float]] = {}   # [group][key] = lag_ms
method_b_xc: dict[str, dict[str, tuple]] = {}  # for xcorr curve plots

for gname, uid_list in groups.items():
    method_b[gname]    = {}
    method_b_xc[gname] = {}

    for key in sd.event_keys:
        sp_mean, _, sp_tax = sd.get_mean_psth(speed_idx, key, SMOOTH_SIG, WINDOW_SEC)

        # Max-normalise each PC PSTH before averaging
        pc_rows = []
        for uid in uid_list:
            m, _, _ = sd.get_mean_psth(uid, key, SMOOTH_SIG, WINDOW_SEC)
            pc_rows.append(norm_max(m))

        if not pc_rows:
            method_b[gname][key] = np.nan
            continue

        grp_mean = np.mean(np.array(pc_rows), axis=0)
        sp_norm  = norm_max(sp_mean)

        lag_ms, lags_ms, xc = xcorr_lag_ms(grp_mean, sp_norm, sp_tax)
        method_b[gname][key]    = lag_ms
        method_b_xc[gname][key] = (lags_ms, xc, grp_mean, sp_norm, sp_tax)

# ── Bar chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(6, n_cond * 1.8), 5))

for gi, (gname, uid_list) in enumerate(groups.items()):
    lags_b = [method_b[gname][k] for k in sd.event_keys]
    offset = (gi - 0.5) * bar_w
    ax.bar(x + offset, lags_b, bar_w,
           color=group_colors[gname], alpha=0.85, label=gname)

ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(sd.event_keys, rotation=30, ha='right')
ax.set_ylabel('Lag (ms)  [+ = PC delayed vs speed]')
ax.set_title('Method B: lag of group-averaged (max-norm) PSTHs')
ax.legend()
plt.tight_layout()
plt.savefig('xcorr_method_b.png', bbox_inches='tight')
plt.show()

### 2e · Cross-correlation curves per group per condition

In [ ]:
n_rows = len(groups)
n_cols = len(sd.event_keys)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(max(4*n_cols, 8), 3.5 * n_rows),
                         squeeze=False, sharey='row')

for ri, (gname, uid_list) in enumerate(groups.items()):
    for ci, key in enumerate(sd.event_keys):
        ax = axes[ri][ci]

        if key not in method_b_xc.get(gname, {}):
            ax.axis('off')
            continue

        lags_ms, xc, grp_mean, sp_norm, tax = method_b_xc[gname][key]
        lag_ms = method_b[gname][key]

        # ---- xcorr curve
        ax.plot(lags_ms, xc, color=group_colors[gname], lw=1.5)
        ax.axvline(0,      color='gray', lw=0.8, ls='--')
        ax.axvline(lag_ms, color='k',   lw=1.0, ls='-',
                   label=f'peak {lag_ms:.0f} ms')
        ax.set_title(f'{gname}\n{key}', fontsize=8)
        ax.set_xlabel('Lag (ms)')
        if ci == 0:
            ax.set_ylabel('Norm. cross-corr')
        ax.legend(fontsize=7)

fig.suptitle('Method B: xcorr curves (group-avg PC vs speed)', fontsize=11)
plt.tight_layout()
plt.savefig('xcorr_curves.png', bbox_inches='tight')
plt.show()

### 2f · Overlay group-averaged PSTHs vs speed per condition

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(max(4*n_cols, 8), 3.5 * n_rows),
                         squeeze=False)

for ri, (gname, uid_list) in enumerate(groups.items()):
    for ci, key in enumerate(sd.event_keys):
        ax = axes[ri][ci]

        if key not in method_b_xc.get(gname, {}):
            ax.axis('off')
            continue

        _, _, grp_mean, sp_norm, tax = method_b_xc[gname][key]

        ax.plot(tax, grp_mean, color=group_colors[gname],
                lw=1.8, label='PC group')
        ax.plot(tax, sp_norm,  color='#FF9800', lw=1.5,
                ls='--', label='Speed')
        ax.axvline(0, color='gray', lw=0.8, ls='--')
        ax.set_title(f'{gname} · {key}', fontsize=8)
        ax.set_xlabel('Time (s)')
        if ci == 0:
            ax.set_ylabel('Normalised (max-norm)')
        if ri == 0 and ci == 0:
            ax.legend(fontsize=8)

fig.suptitle('Group PSTH vs Speed (max-normalised)', fontsize=11)
plt.tight_layout()
plt.savefig('psth_vs_speed.png', bbox_inches='tight')
plt.show()

### 2g · Summary table

In [ ]:
import pandas as pd

rows = []
for key in sd.event_keys:
    for gname in groups:
        rows.append({
            'Condition': key,
            'Group': gname,
            'n_PCs': len(groups[gname]),
            'Lag_A_mean_ms': round(method_a[gname][key][0], 1),
            'Lag_A_sem_ms':  round(method_a[gname][key][1], 1),
            'Lag_B_ms':      round(method_b[gname][key], 1)
                          if not np.isnan(method_b[gname][key]) else np.nan,
        })

df = pd.DataFrame(rows)
df